# AdaptXSS — Evaluation & Benchmarking Notebook

This notebook measures Precision, Recall, F1-score, and Latency for:
- **Baseline A**: Offline Bernoulli NB (bag-of-chars, scikit-learn)
- **Baseline B**: AdaptXSS cold (fresh model, no session updates)
- **Baseline C**: AdaptXSS warm (after 200 training samples)

Dataset: `evaluation/datasets/xss_payloads.csv`

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import BernoulliNB
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score
import time

df = pd.read_csv('datasets/xss_payloads.csv')
print(f"Total samples: {len(df)}")
print(df['label'].value_counts())

In [ ]:
# 80/20 Stratified Split
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)
print(f"Train: {len(train_df)}, Test: {len(test_df)}")

## Baseline A — Offline Bernoulli NB (scikit-learn)

In [ ]:
vec = CountVectorizer(analyzer='char', ngram_range=(2,3), max_features=5000)
X_train = vec.fit_transform(train_df['payload'])
X_test  = vec.transform(test_df['payload'])

clf_offline = BernoulliNB()
t0 = time.time()
clf_offline.fit(X_train, train_df['label'])
train_time = time.time() - t0

t0 = time.time()
preds_offline = clf_offline.predict(X_test)
infer_time = (time.time() - t0) / len(test_df) * 1000  # ms per sample

print("=== Baseline A: Offline BernoulliNB ===")
print(classification_report(test_df['label'], preds_offline))
print(f"Training time: {train_time:.3f}s | Avg inference latency: {infer_time:.3f} ms/sample")

results = {
    'Offline BNB': {
        'precision': precision_score(test_df['label'], preds_offline, pos_label=1),
        'recall':    recall_score(test_df['label'], preds_offline, pos_label=1),
        'f1':        f1_score(test_df['label'], preds_offline, pos_label=1),
        'latency_ms': infer_time,
        'model_size': 'N/A (sklearn in-memory)'
    }
}

## Baseline B/C — AdaptXSS (Cold & Warm)
We simulate DOM mutations using a Python-level feature extractor mirroring `core/src/extractor.js`.

In [ ]:
import re, math

HIGH_RISK_TAGS = {'script','iframe','object','embed','link','meta','base','svg','math'}
MED_RISK_TAGS  = {'img','video','audio','source','track','input','form','button','a'}
DANGEROUS_ATTRS = ['href','src','action','formaction','data','poster']
URL_ATTRS = ['href','src','action','formaction']

def tag_of(html): 
    m = re.match(r'<(\w+)', html.strip())
    return m.group(1).lower() if m else ''

def attrs_of(html):
    return dict(re.findall(r'(\w[\w:-]*)=["\']?([^"\'\s>]*)["\']?', html))

def shannon_entropy(s):
    if not s: return 0
    freq = {}
    for c in s: freq[c] = freq.get(c,0)+1
    ent = 0
    for cnt in freq.values():
        p = cnt/len(s)
        ent -= p * math.log2(p)
    return min(ent/8, 1.0)

def extract_features(html):
    tag = tag_of(html)
    attrs = attrs_of(html)
    f = [0.0]*8
    # f0 tag_risk
    if tag in HIGH_RISK_TAGS: f[0]=1.0
    elif tag in MED_RISK_TAGS: f[0]=0.5
    # f1 attr_delta
    score = sum(0.25 for a in attrs if a in DANGEROUS_ATTRS)
    f[1] = min(score, 1.0)
    # f2 script_injection
    f[2] = 1 if re.search(r'(<script[\s>]|javascript\s*:)', html, re.I) else 0
    # f3 inline_handler
    f[3] = 1 if any(a.startswith('on') for a in attrs) else 0
    # f4 url_anomaly
    for ua in URL_ATTRS:
        val = attrs.get(ua,'').lower().strip()
        if re.match(r'^(javascript|vbscript|data|blob):', val): f[4]=1.0; break
        elif re.search(r'[<>"\']', val): f[4]=0.7
    # f5 data_uri
    if attrs.get('src','').startswith('data:') or attrs.get('href','').startswith('data:'): f[5]=1
    # f6 dom_depth (fixed at 1 for simulation)
    f[6] = 1/20
    # f7 text_entropy
    f[7] = shannon_entropy(html)
    return f

print("Feature extractor ready.")

In [ ]:
import json, math

DEFAULT_STATE = {
    'classCounts': {'malicious':1,'benign':1},
    'featureCounts': {'malicious':[1]*8,'benign':[1]*8},
    'totalSamples': 2
}

def predict_nb(state, features, threshold=0.5):
    log_probs = {}
    for cls in ['malicious','benign']:
        total = state['classCounts'][cls]
        prior = math.log(total / state['totalSamples'])
        likelihood = 0
        for i,fi in enumerate(features):
            present = 1 if fi > 0.5 else 0
            count = state['featureCounts'][cls][i]
            total_cls = total + 8
            p = (count+1)/(total_cls+2)
            likelihood += math.log(p) if present else math.log(1-p)
        log_probs[cls] = prior + likelihood
    max_lp = max(log_probs.values())
    exp_sum = sum(math.exp(lp-max_lp) for lp in log_probs.values())
    mal_prob = math.exp(log_probs['malicious']-max_lp) / exp_sum
    return 'malicious' if mal_prob > threshold else 'benign', mal_prob

def update_nb(state, features, true_label):
    state['classCounts'][true_label] += 1
    state['totalSamples'] += 1
    for i,fi in enumerate(features):
        if fi > 0.5: state['featureCounts'][true_label][i] += 1

print("Online NB ready.")

In [ ]:
import copy, time

label_map = {1:'malicious', 0:'benign'}

# --- COLD model ---
cold_state = copy.deepcopy(DEFAULT_STATE)
cold_preds, cold_lats = [], []
for _, row in test_df.iterrows():
    feats = extract_features(str(row['payload']))
    t0 = time.perf_counter()
    pred, prob = predict_nb(cold_state, feats)
    lat = (time.perf_counter()-t0)*1000
    cold_preds.append(1 if pred=='malicious' else 0)
    cold_lats.append(lat)

# --- WARM model (pre-train on training set) ---
warm_state = copy.deepcopy(DEFAULT_STATE)
for _, row in train_df.iterrows():
    feats = extract_features(str(row['payload']))
    update_nb(warm_state, feats, label_map[row['label']])

warm_preds, warm_lats = [], []
for _, row in test_df.iterrows():
    feats = extract_features(str(row['payload']))
    t0 = time.perf_counter()
    pred, prob = predict_nb(warm_state, feats)
    lat = (time.perf_counter()-t0)*1000
    warm_preds.append(1 if pred=='malicious' else 0)
    warm_lats.append(lat)

true_labels = test_df['label'].tolist()

results['AdaptXSS (cold)'] = {
    'precision': precision_score(true_labels, cold_preds, zero_division=0),
    'recall':    recall_score(true_labels, cold_preds, zero_division=0),
    'f1':        f1_score(true_labels, cold_preds, zero_division=0),
    'latency_ms': np.mean(cold_lats),
    'model_size': '< 10 KB'
}
results['AdaptXSS (warm)'] = {
    'precision': precision_score(true_labels, warm_preds, zero_division=0),
    'recall':    recall_score(true_labels, warm_preds, zero_division=0),
    'f1':        f1_score(true_labels, warm_preds, zero_division=0),
    'latency_ms': np.mean(warm_lats),
    'model_size': '< 10 KB'
}

print("Cold preds:", classification_report(true_labels, cold_preds))
print("Warm preds:", classification_report(true_labels, warm_preds))
print(f"Avg latency COLD: {np.mean(cold_lats):.4f}ms | P99: {np.percentile(cold_lats,99):.4f}ms")
print(f"Avg latency WARM: {np.mean(warm_lats):.4f}ms | P99: {np.percentile(warm_lats,99):.4f}ms")

## Results Table

In [ ]:
res_df = pd.DataFrame(results).T.reset_index()
res_df.columns = ['System','Precision','Recall','F1','Mean Latency (ms)','Model Size']
res_df[['Precision','Recall','F1']] = res_df[['Precision','Recall','F1']].apply(pd.to_numeric)
print(res_df.to_string(index=False))

## Model Drift Experiment
5 simulated sessions × 100 mutations — track F1 per session

In [ ]:
import matplotlib.pyplot as plt

drift_state = copy.deepcopy(DEFAULT_STATE)
f1_per_session = []

all_rows = df.sample(frac=1, random_state=99).reset_index(drop=True)

for sess in range(5):
    batch = all_rows.iloc[sess*100:(sess+1)*100]
    sess_preds, sess_true = [], []
    for _, row in batch.iterrows():
        feats = extract_features(str(row['payload']))
        pred, _ = predict_nb(drift_state, feats)
        sess_preds.append(1 if pred=='malicious' else 0)
        sess_true.append(row['label'])
        update_nb(drift_state, feats, label_map[row['label']])
    f1 = f1_score(sess_true, sess_preds, zero_division=0)
    f1_per_session.append(f1)
    print(f"Session {sess+1}: F1 = {f1:.4f}")

plt.figure(figsize=(8,4))
plt.plot(range(1,6), f1_per_session, marker='o', color='#f97316', linewidth=2)
plt.xlabel('Session Number')
plt.ylabel('F1 Score')
plt.title('AdaptXSS — Model Drift (F1 per Session)')
plt.ylim(0,1)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../paper/figures/model_drift.png', dpi=150)
plt.show()
print("Drift plot saved to paper/figures/model_drift.png")